# `IDP-CV`: Invoice data extraction
**With `Docling` processing, `SentenceTransformer` embeddings and `spaCy` NER**

## Dependencies

In [1]:
# import json
import logging
import os
import sys
import warnings
import pandas as pd

from pathlib import Path

# import spacy
import torch

from docling.datamodel.settings import settings

from run_extraction import process_and_extract_results

/home/crus7ev/miniconda3/envs/idp_cv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Global Logic / Settings Configuration

In [2]:
# Force offline mode for transformers/sentence-transformers
os.environ['TRANSFORMERS_OFFLINE'] = '1'

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
DATA_DIR = Path('./data/example_invoices/file_inputs/')
OUTPUT_DIR = Path('./data/example_invoices/idp-cv_outputs/')

# docling runtime settings
settings.perf.doc_batch_concurrency = 4
settings.perf.doc_batch_size = 4

# Logger configuration for notebooks
logging.basicConfig(level=logging.INFO, format='%(message)s', stream=sys.stdout, force=True)
logging.getLogger('RapidOCR').setLevel(logging.WARNING)
logging.getLogger('docling').setLevel(logging.WARNING)
logging.getLogger('idp_cv').setLevel(logging.DEBUG)
warnings.filterwarnings('ignore', category=UserWarning, module='docling')

## Schema-aware Tables and Summary extraction
- Processes PDFs and document images with `Docling`'s default `DocumentConverter` for OCR, table and layout detection
- Disambiguates table column labels by applying lexical and semantic ranking with schema aliases using `TableFieldMapper`
- Extracts key/value fields from `DoclingDocument` via label ranking with schema and `spaCy` NER validation `DocumentFieldExtractor` 
- Visualises text and table field boxes on page images (**TODO**: labels)

In [3]:
# Run the unified execution procedure
results = process_and_extract_results(input_dir=DATA_DIR, output_dir=OUTPUT_DIR, device=DEVICE)

Load pretrained SentenceTransformer: ibm-granite/granite-embedding-small-english-r2


Batches: 100%|██████████| 1/1 [00:00<00:00, 59.78it/s]

Found 7 files to process in data/example_invoices/file_inputs



[INFO] 2026-03-09 07:29:34,161 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-09 07:29:34,188 [RapidOCR] download_file.py:60: File exists and is valid: /home/crus7ev/miniconda3/envs/idp_cv/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-09 07:29:34,189 [RapidOCR] main.py:53: Using /home/crus7ev/miniconda3/envs/idp_cv/lib/python3.13/site-packages/rapidocr/models/ch_PP-OCRv4_det_infer.onnx
[INFO] 2026-03-09 07:29:34,278 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-03-09 07:29:34,279 [RapidOCR] download_file.py:60: File exists and is valid: /home/crus7ev/miniconda3/envs/idp_cv/lib/python3.13/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-09 07:29:34,280 [RapidOCR] main.py:53: Using /home/crus7ev/miniconda3/envs/idp_cv/lib/python3.13/site-packages/rapidocr/models/ch_ppocr_mobile_v2.0_cls_infer.onnx
[INFO] 2026-03-09 07:29:34,314 [RapidOCR] base.py:22: Using engine_name: onnx

Usage of TableItem.export_to_dataframe() without `doc` argument is deprecated.
Mapping table with columns: ['item', 'desc', 'price', 'qty', 'line total']
Using LexicalMapper with aliases: {'desc': ['description', 'desc', 'item description', 'service', 'product', 'details', 'item'], 'qty': ['quantity', 'qty', 'quant', 'units', 'count'], 'price': ['unit price', 'price', 'rate', 'unit cost', 'net price', 'cost per unit', 'cpu'], 'total': ['gross_worth', 'gross total', 'grand total', 'invoice total', 'total_worth', 'total amount', 'total due', 'balance due', 'total balance', 'total', 'balance', 'balance amount', 'full amount'], 'net': ['net worth', 'net amount', 'net total', 'subtotal', 'sub total', 'amount before tax'], 'tax': ['tax', 'vat', 'gst', 'tax amount', 'vat amount', 'gst amount', 'tax due', 'vat due', 'gst due', 'total tax'], 'tax_rate': ['tax (%)', 'vat (%)', 'gst (%)', 'tax %', 'vat %', 'gst %', 'tax rate', 'vat rate', 'gst rate'], 'shipping': ['shipping', 'delivery', 'freight

...

Batches: 100%|██████████| 1/1 [00:00<00:00, 70.93it/s]

Mapping results: {'desc': {'best': ('item', 1.0, 'lexical'), 'candidates': [('item', 1.0, 'lexical')]}, 'qty': {'best': ('quantity', 1.0, 'lexical'), 'candidates': [('quantity', 1.0, 'lexical'), ('amount', 0.6666666666666667, 'lexical')]}}
Mapped row: {'desc': 'okidata inkjet, durable', 'qty': '2'} to table line: description='okidata inkjet, durable' quantity=2 unit_price=None total_amount=None net_amount=None tax_amount=None tax_rate=None shipping_cost=None.
Skipping row due to validation error: 1 validation error for TableLine
qty
  Input should be a valid integer [type=int_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/int_type
Skipping row due to validation error: 1 validation error for TableLine
qty
  Input should be a valid integer [type=int_type, input_value=None, input_type=NoneType]
    For further information visit https://errors.pydantic.dev/2.12/v/int_type
Skipping row due to validation error: 1 validation e


Batches: 100%|██████████| 1/1 [00:00<00:00, 50.27it/s]

Field Key Mapping Results:

  - Field "receiver" matched key "Bill To:" (score: 1.00 [lexical]) at 1 locations: group[0]
    - Candidates considered:
    'Bill To:' (1.00 [lexical]), 'Bill' (0.50 [lexical]), 'Huston' (0.44 [lexical]), 'Crewe, England,' (0.33 [lexical]), 'Crewe,' (0.29 [lexical])

  - Field "order_id" matched key "Order ID :" (score: 0.90 [lexical]) at 1 locations: group[1]
    - Candidates considered:
    'Order ID :' (0.90 [lexical]), 'Order ID' (0.89 [lexical]), 'To:' (0.67 [lexical]), 'Mode:' (0.50 [lexical]), 'ID' (0.50 [lexical])

  - Field "receiver_addr" matched key "Ship To:" (score: 1.00 [lexical]) at 1 locations: group[0]


    - Candidates considered:
    'Ship To:' (1.00 [lexical]), 'Ship Mode:' (0.70 [lexical]), 'Ship' (0.50 [lexical]), 'Apr' (0.40 [lexical]), 'business!' (0.38 [lexical])

  - Field "inv_no" matched key "INVOICE" (score: 0.88 [lexical]) at 1 locations: group[2]
    - Candidates considered:
    'INVOICE' (0.88 [lexical]), 'United' (0.38 [lexical]), 'Thanks for' (0.30 [lexical]), 'Kingdom' (0.29 [lexical]), 'United Kingdom' (0.29 [lexical])

  - Field "total" matched key "Balance Due:" (score: 1.00 [lexical]) at 1 locations: group[0]
    - Candidates considered:
    'Balance Due:' (1.00 [lexical]), 'Balance' (0.88 [lexical]), 'England, United' (0.33 [lexical])

  - Field "date" matched key "Date:" (score: 1.00 [lexical]) at 1 locations: group[0]
    - Candidates considered:
    'Date:' (1.00 [lexical]), 'Due:' (0.60 [lexical]), 'Notes:' (0.50 [lexical]), 'Apr 13 2012' (0.21 [lexical])

  => Unmatched Fields: due, subtotal, tax_amount, tax_rate, shipping, issuer, issuer_tax, receiver_tax,

In [4]:
# Visualization and Formatting for Summaries
summary_data = []
for res in results:
    if 'summary' in res:
        entry = res['summary'].copy()
        entry['Document'] = res['Document']
        summary_data.append(entry)

if summary_data:
    df = pd.DataFrame(summary_data)
    # Ensure 'Document' is the first column
    cols = ['Document'] + [c for c in df.columns if c != 'Document']
    df = df[cols].set_index('Document')

    num_cols = df.select_dtypes(include='number').columns
    display(df.style.format({c: '{:.2f}' for c in num_cols}, na_rep='None'))
else:
    print('No results found.')

,invoice_number,order_id,invoice_date,due_date,total_amount,net_amount,tax_amount,tax_rate,shipping_cost,vendor_name,buyer_name,vendor_tax_id,buyer_tax_id,vendor_address,buyer_address
Document,,,,,,,,,,,,,,,
inv1,INV-6059,None,2025-07-25,None,9251.14,8621.10,None,None,None,Oceanic Corp.,Acme LLC,None,None,"101 Marine Drive Miami, FL 33101","55 Industrial Rd Chicago, IL 60601"
inv2,None,None,None,None,5175.49,4824.45,337.71,0.07,13.33,Skyline Solutions,None,None,None,"456 Cloud St Austin, TX 73301",None
inv3,37535203,None,2015-07-13,None,475.75,432.50,43.25,0.10,None,Garcia-Burton,Miranda-Nielsen,916-83-8601,989-84-4547,"0340MoranPort Pattonborough,Wl65859","8668RangelStravenueSuite392 PortKristen,SD88510"
inv4,13310164,None,2020-01-16,None,18749.50,17045.00,1704.50,0.10,None,MontgomeryLLC,Campbell-Flores,924-83-8802,None,"75081David Corners LakeKellyview,NH 03078",34813DavisViews
inv5,1234NA1,PO-65652345,2025-07-08,2025-07-16,36.44,30.37,6.07,0.20,None,John Doe Electronics ltd.,Jane Doe,None,None,None,"123, Apple str."
inv6,INV-8114,PO-49113,2025-07-05,None,10158.78,9470.66,662.95,0.07,25.17,Oceanic Corp.,Jane Smith,None,None,"101 Marine Drive Miami, FL 33101","300 Elm St Seattle, WA 98101"
inv7,16892,ES-2012-JH15820139-41012,2012-04-13,None,1337.97,1250.04,None,None,87.93,SuperStore,John Huston,None,None,None,"Crewe, England, United Kingdom"


In [5]:
[item.model_dump(exclude_none=True) for res in results if 'line_items' in res for item in res['line_items'][0]]

[{'description': 'monitor',
  'quantity': 7,
  'unit_price': 346.87,
  'net_amount': 2428.09},
 {'description': 'desk chair',
  'quantity': 10,
  'unit_price': 488.03,
  'net_amount': 4880.3},
 {'description': 'laptop',
  'quantity': 7,
  'unit_price': 187.53,
  'net_amount': 1312.71},
 {'description': 'keyboard', 'quantity': 4, 'total_amount': 436.64},
 {'description': 'desk chair', 'quantity': 7, 'total_amount': 2377.34},
 {'description': 'software license', 'quantity': 5, 'total_amount': 272.15},
 {'description': 'laptop', 'quantity': 4, 'total_amount': 1738.32},
 {'description': 'q&aaday:5-yearjournal',
  'quantity': 3,
  'unit_price': 4.49,
  'total_amount': 14.82,
  'net_amount': 13.47,
  'tax_rate': 10.0},
 {'description': "poorcharlie'salmanackthewit andwisdomofcharlest munger3rded.brandnew",
  'quantity': 5,
  'unit_price': 64.95,
  'total_amount': 357.23,
  'net_amount': 324.75,
  'tax_rate': 10.0},
 {'description': "againstallodds:theuntold storyofcanada'sunlikely hockey her

In [6]:
[item.model_dump(exclude_none=True) for res in results if 'summary_table' in res for item in res['summary_table'][0]]

[{'total_amount': 475.75,
  'net_amount': 432.5,
  'tax_amount': 43.25,
  'tax_rate': 10.0},
 {'total_amount': 475.75, 'net_amount': 432.5, 'tax_amount': 43.25},
 {'total_amount': 18749.5,
  'net_amount': 17045.0,
  'tax_amount': 1704.5,
  'tax_rate': 10.0},
 {'total_amount': 18749.5, 'net_amount': 17045.0, 'tax_amount': 1704.5},
 {'total_amount': 1337.97, 'net_amount': 1250.04, 'shipping_cost': 87.93}]

# END